In [1]:
# ============================================================
# YOLOv26-medium (yolo26m)  |  TRAIN + DANH GIA TEST trong 1 CELL
# Truoc khi chay:
#   (1) Add Input: dataset chua YOLO_Final_Dataset  ->  sua KAGGLE_DATA_PATH ben duoi
#   (2) Settings: Accelerator = GPU T4 x2   |   Internet = On
# ============================================================
!pip install -q ultralytics

from ultralytics import YOLO
import glob, os

# -------- CAU HINH --------
KAGGLE_DATA_PATH = '/kaggle/input/datasets/tungtungtungtung/middd-yolo-26/YOLO_Final_Dataset'  # <-- SUA cho khop
WEIGHTS  = 'yolo26m.pt'
RUN_NAME = 'yolo26m_telecom'
BATCH    = 8

# tao dataset.yaml
yaml_content = f"""
path: {KAGGLE_DATA_PATH}
train: train/images
val: val/images
test: test/images

nc: 6
names: ['anten-4G', 'anten-5G', 'none', 'rrh', 'rru', 'viba']
"""
YAML_PATH = '/kaggle/working/dataset.yaml'
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content.strip())

# -------- TRAIN --------
model = YOLO(WEIGHTS)
model.train(
    data=YAML_PATH, imgsz=640, device=[0, 1], batch=BATCH, workers=8, amp=True,
    epochs=150, patience=25, pretrained=True, optimizer='auto', lr0=0.01,
    cos_lr=True, close_mosaic=10, seed=42, deterministic=True,
    project='telecom_vision_project', name=RUN_NAME, save_period=10,
)
print('🎉 Huan luyen hoan tat! Model luu trong runs/detect/telecom_vision_project/' + RUN_NAME)

# -------- DANH GIA TEST (boc try/except -> KHONG bao gio lam hong run khi save) --------
try:
    cands = glob.glob(f'/kaggle/working/**/{RUN_NAME}*/weights/best.pt', recursive=True)
    best_path = max(cands, key=os.path.getmtime)
    print('Best weights:', best_path)
    mt = YOLO(best_path).val(data=YAML_PATH, split='test', imgsz=640)
    print(f'>>> Test mAP50-95: {mt.box.map:.4f} | mAP50: {mt.box.map50:.4f} | mAP75: {mt.box.map75:.4f}')
except Exception as e:
    print('Bo qua danh gia test (model da train van duoc luu an toan):', repr(e))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic